# Uso de GPU

In [22]:
import tensorflow as tf

print("TF version:", tf.__version__)
print("Built with CUDA:", tf.test.is_built_with_cuda())

gpus = tf.config.list_physical_devices("GPU")
print("GPUs:", gpus)

if gpus:
    for i, g in enumerate(gpus):
        details = tf.config.experimental.get_device_details(g)
        print(f"GPU {i} details:", details)
else:
    print("Nenhuma GPU visível para o TensorFlow.")

I0000 00:00:1780078757.354344 1242472 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


TF version: 2.21.0
Built with CUDA: True
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]
GPU 0 details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4090'}
GPU 1 details: {'compute_capability': (8, 9), 'device_name': 'NVIDIA GeForce RTX 4090'}


# Análise demográfica

In [23]:
import pandas as pd

df_ext = pd.read_csv("image_data_sMCI_pMCI_extremos.csv")

# 1 linha por paciente (metadados do conjunto)
pt = (
    df_ext.sort_values(["ID_PT", "MRI_DATE", "ID_IMG"])
    .groupby("ID_PT", as_index=False)
    .agg(
        GROUP=("GROUP", "first"),
        SEX=("SEX", "first"),
        n_linhas=("ID_IMG", "size"),
    )
)

# conjuntos = pacientes com exatamente 3 linhas
pt = pt[pt["n_linhas"] == 3].copy()
pt["n_conjuntos"] = 1  # 1 conjunto por paciente válido

# --- totais ---
n_conjuntos_total = len(pt)
n_linhas_total = int(pt["n_linhas"].sum())
print("Conjuntos (pacientes com 3 linhas):", n_conjuntos_total)
print("Linhas totais:", n_linhas_total)
print("Checagem linhas/3:", n_linhas_total / 3)

# --- por GROUP ---
print("\nPor GROUP:")
print(pt["GROUP"].value_counts().sort_index())
# ou só contagem de conjuntos:
# print(pt.groupby("GROUP")["n_conjuntos"].sum())

# --- por SEX ---
print("\nPor SEX:")
print(pt["SEX"].value_counts().sort_index())

# --- GROUP × SEX ---
print("\nGROUP × SEX:")
print(pd.crosstab(pt["GROUP"], pt["SEX"], margins=True))

Conjuntos (pacientes com 3 linhas): 525
Linhas totais: 1575
Checagem linhas/3: 525.0

Por GROUP:
GROUP
pMCI    128
sMCI    397
Name: count, dtype: int64

Por SEX:
SEX
F    222
M    303
Name: count, dtype: int64

GROUP × SEX:
SEX      F    M  All
GROUP               
pMCI    54   74  128
sMCI   168  229  397
All    222  303  525


In [24]:
df_ext.groupby("GROUP")["ID_PT"].nunique()

GROUP
pMCI    128
sMCI    397
Name: ID_PT, dtype: int64

In [25]:
dist_pacientes = (
    df_ext.drop_duplicates(subset=["ID_PT", "GROUP"])
    .groupby(["GROUP", "SEX"])["ID_PT"]
    .nunique()
    .unstack(fill_value=0)
)

dist_pacientes

SEX,F,M
GROUP,,
pMCI,54,74
sMCI,168,229


In [26]:
pts = df_ext.groupby(["GROUP", "ID_PT"], as_index=False)["AGE"].min()

for g in ["sMCI", "pMCI"]:
    a = pts.loc[pts["GROUP"] == g, "AGE"]
    print(
        f"{g}: n={a.count()}, "
        f"min={a.min():.1f}, max={a.max():.1f}, "
        f"media={a.mean():.2f}, desvio={a.std():.2f}"
    )

sMCI: n=397, min=55.0, max=91.0, media=73.52, desvio=7.48
pMCI: n=128, min=57.0, max=89.0, media=75.04, desvio=7.01


In [27]:
adnimerge = pd.read_csv("csvs/adnimerged.csv")
df_ext = pd.read_csv("image_data_sMCI_pMCI_extremos.csv")

cols = ["ID_IMG", "MMSE_SCORE", "CDR_GLOBAL", "ADAS_SCORE", "FAQ_SCORE"]

adni_sub = adnimerge.loc[:, cols].copy()
adni_sub["ID_IMG"] = adni_sub["ID_IMG"].astype(str).str.strip()
adni_sub = adni_sub.drop_duplicates(subset=["ID_IMG"], keep="last")

df_ext["ID_IMG"] = df_ext["ID_IMG"].astype(str).str.strip()
df_ext = df_ext.merge(adni_sub, on="ID_IMG", how="left", validate="many_to_one")

In [28]:
pts = (
    df_ext.sort_values(["GROUP", "ID_PT", "MRI_DATE"])
    .drop_duplicates(subset=["GROUP", "ID_PT"], keep="first")
)

pts.groupby("GROUP")[["MMSE_SCORE", "FAQ_SCORE", "CDR_GLOBAL", "ADAS_SCORE"]].agg(["mean", "std", "count"])

MMSE_SCORE                 FAQ_SCORE                 CDR_GLOBAL  \
            mean       std count      mean       std count       mean   
GROUP                                                                   
pMCI   26.039062  2.249768   128  6.453125  4.619131   128   0.507812   
sMCI   27.758186  1.822132   397  2.503778  3.573628   397   0.497481   

                      ADAS_SCORE                  
            std count       mean       std count  
GROUP                                             
pMCI   0.062253   128  14.382891  5.055354   128  
sMCI   0.035444   397   9.491788  3.917522   397

# Modelagem

In [1]:
# Nested CV 5×5 + SVM + downsampling (só no treino)
# 1 paciente = 1 linha (média das 60 ROIs) | sMCI=0, pMCI=1 | teste NUNCA balanceado

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

METRIC_COLS = [
    "auc", "auc_pr", "bal_acc", "mcc",
    "sens_pMCI", "spec_sMCI", "prec_pMCI", "f1_pMCI",
]

# --- dados ---
PATH = "csvs/abordagem_4_sMCI_pMCI_extremos/all_features_merge_clean.csv"
df = pd.read_csv(PATH).sort_values("ID_PT")
meta = {"ID_PT", "GROUP", "SEX"}
cols = [c for c in df.columns if c not in meta]

X, y = [], []
for _, g in df.groupby("ID_PT", sort=False):
    if len(g) != 60:
        continue
    X.append(g[cols].to_numpy(float).mean(axis=0))
    y.append(int(g["GROUP"].iloc[0]))
X, y = np.vstack(X), np.array(y)
print(f"{len(y)} pacientes | sMCI={(y==0).sum()} | pMCI={(y==1).sum()}")


def downsample(X, y, seed):
    """Mesmo nº de pacientes sMCI e pMCI (N = minoritária)."""
    rng = np.random.default_rng(seed)
    i_smci = np.where(y == 0)[0]
    i_pmci = np.where(y == 1)[0]
    n = min(len(i_smci), len(i_pmci))
    idx = np.r_[rng.choice(i_smci, n, replace=False), rng.choice(i_pmci, n, replace=False)]
    rng.shuffle(idx)
    return X[idx], y[idx]


def fold_metrics(y_true, scores, pred) -> dict:
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {
        "auc": float(roc_auc_score(y_true, scores)),
        "auc_pr": float(average_precision_score(y_true, scores)),
        "bal_acc": float(balanced_accuracy_score(y_true, pred)),
        "mcc": float(matthews_corrcoef(y_true, pred)),
        "sens_pMCI": float(tp / (tp + fn)) if (tp + fn) else float("nan"),
        "spec_sMCI": float(tn / (tn + fp)) if (tn + fp) else float("nan"),
        "prec_pMCI": float(precision_score(y_true, pred, pos_label=1, zero_division=0)),
        "f1_pMCI": float(f1_score(y_true, pred, pos_label=1, zero_division=0)),
    }


SEED = 42
K_OUT, K_IN = 5, 5
C_GRID = [1e-3, 0.01, 0.1, 1.0, 10.0]


def svm(C):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LinearSVC(C=C, max_iter=10000, dual="auto", random_state=SEED)),
    ])


results = []
outer = StratifiedKFold(K_OUT, shuffle=True, random_state=SEED)

for fold, (tr, te) in enumerate(outer.split(X, y), start=1):
    best_C, best_inner_auc = C_GRID[0], -1.0
    inner = StratifiedKFold(K_IN, shuffle=True, random_state=SEED + fold)

    for C in C_GRID:
        inner_aucs = []
        for in_tr, in_va in inner.split(X[tr], y[tr]):
            Xb, yb = downsample(X[tr][in_tr], y[tr][in_tr], SEED + fold + int(C * 1e4))
            model = svm(C)
            model.fit(Xb, yb)
            inner_aucs.append(roc_auc_score(y[tr][in_va], model.decision_function(X[tr][in_va])))
        if np.mean(inner_aucs) > best_inner_auc:
            best_inner_auc, best_C = float(np.mean(inner_aucs)), C

    Xb, yb = downsample(X[tr], y[tr], SEED + fold)
    model = svm(best_C)
    model.fit(Xb, yb)
    scores = model.decision_function(X[te])
    pred = model.predict(X[te])

    results.append({
        "fold": fold,
        "best_C": best_C,
        "n_train_bal": len(yb),
        "n_test_pMCI": int(y[te].sum()),
        **fold_metrics(y[te], scores, pred),
    })

results = pd.DataFrame(results)
display(results)

print("\nResumo (média ± desvio entre folds):")
for col in METRIC_COLS:
    print(f"  {col:12s}: {results[col].mean():.3f} ± {results[col].std():.3f}")


525 pacientes | sMCI=397 | pMCI=128


/home/pgirardi/.pyenvs/colab/lib/python3.12/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/pgirardi/.pyenvs/colab/lib/python3.12/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/pgirardi/.pyenvs/colab/lib/python3.12/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/pgirardi/.pyenvs/colab/lib/python3.12/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/pgirardi/.pyenvs/colab/lib/python3.12/site-packages/sklearn/svm/_base.py:1237: ConvergenceWarning: Liblinear failed to converge, increase the number of iterations.
  warnings.warn(
/home/pgirardi/.pyenvs/colab/lib/python3.12/site-packages/sklearn

,fold,best_C,n_train_bal,n_test_pMCI,auc,auc_pr,bal_acc,mcc,sens_pMCI,spec_sMCI,prec_pMCI,f1_pMCI
0,1,0.001,204,26,0.753165,0.524658,0.661879,0.296441,0.576923,0.746835,0.428571,0.491803
1,2,0.001,204,26,0.683544,0.362333,0.643622,0.248248,0.692308,0.594937,0.360000,0.473684
2,3,0.010,204,26,0.780428,0.539534,0.700097,0.372090,0.615385,0.784810,0.484848,0.542373
3,4,0.010,206,25,0.739500,0.447164,0.646250,0.250082,0.680000,0.612500,0.354167,0.465753
4,5,0.010,206,25,0.640000,0.454626,0.632500,0.227487,0.640000,0.625000,0.347826,0.450704



Resumo (média ± desvio entre folds):
  auc         : 0.719 ± 0.057
  auc_pr      : 0.466 ± 0.071
  bal_acc     : 0.657 ± 0.026
  mcc         : 0.279 ± 0.058
  sens_pMCI   : 0.641 ± 0.047
  spec_sMCI   : 0.673 ± 0.087
  prec_pMCI   : 0.395 ± 0.060
  f1_pMCI     : 0.485 ± 0.035
